<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main">Spatial Data Science Approaches to Wildfire Severity Modeling</h1>
  <hr class="title-rule">
  <h2 class="title-sub">A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfire</h2>
</div>

## Module 11: *Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> 1. **
> 2. **
> 3. **
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions
from src.data_utils import create_interactions

# A space saving function to rank interactions
from src.data_utils import rank_variables_by_correlation

---
### Third Party Dependencies

In [ ]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling prep
from sklearn.preprocessing import MinMaxScaler

---

###  Load Data

In [ ]:
X = pd.read_csv('../data/processed/X.csv')
y = pd.read_csv('../data/processed/y.csv').squeeze()  # Load as Series
details = pd.read_csv('../data/processed/details.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/best_strategy.csv')
model_parameters = pd.read_csv('../data/processed/model_parameters.csv', index_col=0)

## 1. Build Models

In [7]:
model_parameters = pd.read_csv('../data/processed/model_parameters.csv', index_col=0)

In [8]:

# data was stored as string in dataframe, convert to int for building models
RF_parameters = model_parameters.loc['RandomForest'].dropna().to_dict()
XGB_parameters = model_parameters.loc['XGBoost'].dropna().to_dict()
LGBM_parameters = model_parameters.loc['LGBM'].dropna().to_dict()
# save float data

optimal_learning_rate = XGB_parameters['learning_rate']
optimal_learning_rate_lgbm = LGBM_parameters['learning_rate']


In [9]:
def convert_to_int(d):
    return {k: int(float(v)) if str(v).replace('.', '', 1).isdigit() else v for k, v in d.items()}


# data was stored as string in dataframe, convert to int for building models
RF_parameters = convert_to_int(RF_parameters)
XGB_parameters = convert_to_int(XGB_parameters)
LGBM_parameters = convert_to_int(LGBM_parameters)


# assign floats back to key variables
XGB_parameters['learning_rate'] = optimal_learning_rate
LGBM_parameters['learning_rate'] = optimal_learning_rate_lgbm
LGBM_parameters['verbose'] = -1

In [10]:
display(RF_parameters)
display(XGB_parameters)
display(LGBM_parameters)

{'n_estimators': 150,
 'max_depth': 20,
 'min_samples_split': 5,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

{'n_estimators': 200,
 'max_depth': 6,
 'objective': 'multi:softmax',
 'num_class': 3,
 'learning_rate': 0.4,
 'verbosity': 0}

{'n_estimators': 150,
 'max_depth': -1,
 'class_weight': 'balanced',
 'learning_rate': 50.0,
 'num_leaves': 255,
 'verbose': -1}

In [11]:
# Build Final tuned models
optimum_xgb_model = xgb.XGBClassifier(**XGB_parameters)
optimum_rf = RandomForestClassifier(**RF_parameters)
optimum_lgbm_model = LGBMClassifier(**LGBM_parameters)

In [ ]:
EXPLICIT_VALIDATION_TEMPORAL = [
    "Year",
    "2-Year Avg Fires"
]


In [ ]:
EXPLICIT_VALIDATION_HUMAN = [
    "median_income",
    "road_length_meters",
    "road_density",
    "power_line_meters",
    "power_line_density",
    "total_housing",
    "housing_density",
    "total_population",
    "population_density"
]


In [ ]:
EXPLICIT_VALIDATION_WUI = [
    "influence_zone",
    "interface_zone",
    "intermix_zone"
]


In [ ]:
EXPLICIT_VALIDATION_ECOREGION = [
    # Province
    "dominant_province_description_American Semi-Desert and Desert",
    "dominant_province_description_California Coastal Chaparral Forest and Shrub",
    "dominant_province_description_California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow",
    "dominant_province_description_California Coastal Steppe-Mixed Forest-Redwood Forest",
    "dominant_province_description_California Dry Steppe",
    "dominant_province_description_Intermountain Semi-Desert",
    "dominant_province_description_Intermountain Semi-Desert and Desert",
    "dominant_province_description_Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow",

    # Section
    "dominant_section_description_Central California Coast",
    "dominant_section_description_Central Valley Coast Ranges",
    "dominant_section_description_Colorado Desert",
    "dominant_section_description_Great Valley",
    "dominant_section_description_Klamath Mountains",
    "dominant_section_description_Modoc Plateau",
    "dominant_section_description_Mojave Desert",
    "dominant_section_description_Mono",
    "dominant_section_description_Northern California Coast",
    "dominant_section_description_Northern California Coast Ranges",
    "dominant_section_description_Northern California Interior Coast Ranges",
    "dominant_section_description_Northwestern Basin and Range",
    "dominant_section_description_Sierra Nevada",
    "dominant_section_description_Sierra Nevada Foothills",
    "dominant_section_description_Sonoran Desert",
    "dominant_section_description_Southeastern Great Basin",
    "dominant_section_description_Southern California Coast",
    "dominant_section_description_Southern California Mountains and Valleys",
    "dominant_section_description_Southern Cascades"
]



In [ ]:
DEFENSIBLE_WEATHER_FUEL = [
    "Burning Index",
    "Energy Release Component",
    "Actual Evapotranspiration",
    "100-hour Dead Fuel Moisture",
    "1000-hour Dead Fuel Moisture",
    "Precipitation",
    "Maximum Relative Humidity",
    "Minimum Relative Humidity",
    "Specific Humidity",
    "Solar Radiation",
    "Daily Minimum Air Temperature",
    "Daily Maximum Air Temperature",
    "Vapor Pressure Deficit",
    "Wind Speed",
    "Standardized Precipitation Index 30-Day",
    "Standardized Precipitation Index 180-Day",
    "Standardized Precipitation Evapotranspiration Index 30-Day",
    "Standardized Precipitation Evapotranspiration Index 90-Day",
    "Standardized Precipitation Evapotranspiration Index 180-Day",
    "Palmer Drought Severity Index",

    # 7-day aggregates
    "Vapor Pressure Deficit 7 Day Avg",
    "Precipitation 7 Day Avg",
    "Solar Radiation 7 Day Avg",
    "Daily Maximum Air Temperature 7 Day Avg",
    "Daily Minimum Air Temperature 7 Day Avg",
    "Maximum Relative Humidity 7 Day Avg",
    "Minimum Relative Humidity 7 Day Avg",
    "Wind Speed 7 Day Avg",

    "Santa_Ana_Score"
]


In [ ]:
DEFENSIBLE_TOPOGRAPHY = [
    "elevation_mean",
    "elevation_range",
    "elevation_std",
    "slope_mean",
    "slope_max",
    "slope_range",
    "slope_std",
    "northness_mean",
    "eastness_mean"
]


In [ ]:
DEFENSIBLE_LAND_COVER = [
    "forest_percent",
    "developed_percent",
    "other_percent",
    "shrub_grass_percent",
    "wetlands_percent"
]


In [ ]:
DEFENSIBLE_SEASONAL = [
    "Season_Fall",
    "Season_Spring",
    "Season_Suumer",
    "Season_Winter"
]


In [ ]:
DEFENSIBLE_INTERACTIONS = [
    "Wind Speed_x_100-hour Dead Fuel Moisture",
    "Vapor Pressure Deficit_x_Solar Radiation",
    "Precipitation_x_1000-hour Dead Fuel Moisture",
    "northness_mean_x_Daily Maximum Air Temperature",
    "road_density_x_forest_percent",
    "power_line_density_x_total_housing",
    "2-Year Avg Fires_x_Precipitation 7 Day Avg",
    "Wind Speed_x_slope_mean",
    "Wind Speed_x_slope_max",
    "Wind Speed_x_northness_mean",
    "Wind Speed_x_eastness_mean",
    "Wind Speed_x_elevation_mean",
    "Wind Speed_x_elevation_range",
    "Wind Speed 7 Day Avg_x_slope_mean",
    "Wind Speed 7 Day Avg_x_slope_max",
    "Wind Speed 7 Day Avg_x_northness_mean",
    "Wind Speed 7 Day Avg_x_eastness_mean",
    "Wind Speed 7 Day Avg_x_elevation_mean",
    "Wind Speed 7 Day Avg_x_elevation_range"
]


Baseline model → DEFENSIBLE_FEATURES
Full model → DEFENSIBLE_FEATURES + EXPLICIT_VALIDATION_FEATURES
Ablation runs → drop one explicit group at a time

In [ ]:
Baseline_X = X[]

## 2. Train Models

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14)

In [13]:
#X_train_balanced, y_train_balanced = apply_balancing('RF', best_strategy, X_train, y_train)
optimum_rf.fit(X_train, y_train)

#X_train_balanced, y_train_balanced = apply_balancing('LGBM', best_strategy, X_train, y_train)
optimum_lgbm_model.fit(X_train, y_train)

#X_train_balanced, y_train_balanced = apply_balancing('XGB', best_strategy, X_train, y_train)
optimum_xgb_model.set_params(verbosity=0)
optimum_xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.4, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None, num_class=3,
              num_parallel_tree=None, ...)